In [ ]:
cases_raw = session.table(f"{DB}.{RAW}.STREAM_T_CASES").filter(
    col("METADATA$ACTION") == "INSERT"
).drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID").cache_result()
print("Stream read complete")

In [ ]:
ind_cols = ["RESTRICTED_IND", "CASE_ACCESS_IND"]
code_cols = ["CASE_TYPE", "CURRENT_CASE_STATUS_CODE", "RESTRICTION_REASON_CODE"]
desc_cols = ["CASE_NAME", "CASE_CMNT", "IMPORTANT_OBSERVATIONS", "RESTRICTION_REASON_CMNT"]
user_cols = ["ADD_USER", "MOD_USER"]
trim_cols = ["UC_CASE_NAME", "ASSIST_CASE_NUM", "ARCHIVE_FILE_NAME", "SOURCE_FILE_NAME"]

all_cols = [c.name for c in cases_raw.schema.fields]
select_exprs = []

for c in all_cols:
    if c == "LOAD_TIMESTAMP":
        select_exprs.append(col(c).alias("RAW_LOAD_TIMESTAMP"))
    elif c in ind_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("N")).otherwise(trim(col(c))), lit("N")).alias(c))
    elif c in code_cols:
        select_exprs.append(upper(trim(col(c))).alias(c))
    elif c in desc_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("NA")).otherwise(trim(col(c))), lit("NA")).alias(c))
    elif c in user_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("SYSTEM")).otherwise(trim(col(c))), lit("SYSTEM")).alias(c))
    elif c in trim_cols:
        select_exprs.append(trim(col(c)).alias(c))
    else:
        select_exprs.append(col(c))

cases_clean = cases_raw.select(select_exprs)
print("Transformations applied")

In [ ]:
valid_cases = cases_clean.filter(col("CAS_ID").is_not_null())
invalid_cases = cases_clean.filter(col("CAS_ID").is_null()).with_column("ERROR_REASON", lit("CAS_ID_NULL"))
print("Valid/invalid split done")

In [ ]:
checksum_columns = [
    ("CAS_ID", "number"), ("CASE_TYPE", "string"), ("CASE_NAME", "string"),
    ("UC_CASE_NAME", "string"), ("RESTRICTED_IND", "string"), ("CASE_ACCESS_IND", "string"),
    ("ASSIST_CASE_NUM", "string"), ("CASE_CMNT", "string"), ("IMPORTANT_OBSERVATIONS", "string"),
    ("RESTRICTION_REASON_CODE", "string"), ("RESTRICTION_REASON_CMNT", "string"),
    ("CURRENT_CASE_STATUS_CODE", "string"), ("ARCHIVE_FILE_NAME", "string"),
    ("ADD_USER", "string"), ("MOD_USER", "string"),
    ("UNIT_ORGN_ID", "number"), ("AREA_ORGN_ID", "number"), ("REGION_ORGN_ID", "number"),
    ("PERSON_PERSON_REQUESTS_ID", "number"), ("EXPECTED_CLOSE_DATE", "timestamp"),
    ("CURRENT_CASE_STATUS_DATE", "timestamp"), ("RESTRICTION_DATE", "timestamp"),
    ("ARCHIVE_DATE", "timestamp")
]

checksum_exprs = []
for col_name, col_type in checksum_columns:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

cases_final = valid_cases.with_column(
    "CHECKSUM", sha2(concat_ws(lit("|"), *checksum_exprs), 256)
).with_column("STAGING_LOADED_TIMESTAMP", current_timestamp())

cases_final.write.save_as_table(f"{DB}.{STG}.{STG_CASES}", mode="truncate")
print(f"CASES loaded into {STG}.{STG_CASES}")

In [ ]:
invalid_count = invalid_cases.count()

if invalid_count > 0:
    invalid_cases.select(
        lit("T_CASES").alias("TABLE_NAME"),
        col("ERROR_REASON"),
        col("SOURCE_FILE_NAME"),
        col("RAW_LOAD_TIMESTAMP"),
    ).write.save_as_table(f"{DB}.{STG}.{INVALID_RECORDS}", mode="append")
    print(f"Invalid records saved: {invalid_count}")
else:
    print("No invalid records")

In [ ]:
rows_processed = session.table(f"{DB}.{STG}.{STG_CASES}").count()
status = 'SUCCESS' if invalid_count == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    str(uuid.uuid4()), 'STG_CASES_LOAD', 'STAGING',
    datetime.now(), datetime.now(),
    rows_processed, invalid_count, status, None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status, 'STG_CASES_LOAD', 'STAGING',
    f'CASES staging completed. Rows: {rows_processed}, Failed: {invalid_count}'
)
print(f"Done | Valid: {rows_processed} | Invalid: {invalid_count}")